# Academic Research Assistant demo
Select the **Academic Research Assistant (.venv)** kernel. Place text based PDFs in `data/papers/` and start Ollama before running all cells.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from app.config import get_settings
from app.llm import check_connection
from app.rag.vector_store import VectorStore
from app.agents.query_agent import QueryAgent
from app.agents.retrieval_agent import RetrievalAgent
from app.agents.summary_agent import SummaryAgent
from app.agents.fact_check_agent import FactCheckAgent
from app.agents.writer_agent import WriterAgent
settings = get_settings()
print(settings.llm_provider, settings.ollama_model if settings.llm_provider == 'ollama' else settings.openrouter_model)

In [ ]:
print(await check_connection(settings))

In [ ]:
store = VectorStore(settings)
print(store.ingest())
print('Indexed papers:', len(store.papers()))

In [ ]:
# Edit this question to match the papers you placed in data/papers/.
question = 'What do the indexed papers report about federated learning and patient records?'
plan = await QueryAgent(settings).run(question)
plan.model_dump()

In [ ]:
passages = await RetrievalAgent(settings, store).run(plan)
[(p.paper_title, p.page_number, p.chunk_id, round(p.relevance_score, 3)) for p in passages]

In [ ]:
summary = await SummaryAgent(settings).run(question, passages)
summary.model_dump()

In [ ]:
checked = await FactCheckAgent(settings).run(summary, passages)
[(c.text, c.status.value, c.evidence_ids) for c in checked.claims]

In [ ]:
report = await WriterAgent(settings).run(question, checked, passages)
from IPython.display import Markdown, display
display(Markdown(report.markdown))